In [ ]:
#@title 儲存格 1：導入
import os
from datetime import datetime
from etils import epath
import functools
from IPython.display import HTML, clear_output
from typing import Any, Dict, Sequence, Tuple, Union
from ml_collections import config_dict

import jax
from jax import numpy as jp
import numpy as np
from flax import struct
from matplotlib import pyplot as plt
import mediapy as media

import mujoco
from mujoco import mjx

from brax import envs
from brax import math
from brax.envs.base import Env, State
from brax.training.agents.ppo import train as ppo
from brax.training.agents.ppo import networks as ppo_networks
from brax.io import mjcf

print("所有函式庫已導入。")

所有函式庫已導入。


In [83]:
from mujoco import viewer

viewer.launch_from_path("./scene_mjx2.0.xml")

In [ ]:
#@title 儲存格 2：常數與配置

# --- Pupper 常數定義 ---
class _PupperConstants:
    def __init__(self):
        self.PUPPER_ROOT_PATH = epath.Path(os.getcwd())
        self.DEFAULT_XML = self.PUPPER_ROOT_PATH / "scene_mjx2.0.xml"
        self.ROOT_BODY = "torso"
        self.FEET_SITES = ["foot_front_right", "foot_front_left", "foot_hind_right", "foot_hind_left"]
        self.FEET_GEOMS = ["foot_front_right_collision", "foot_front_left_collision", "foot_hind_right_collision", "foot_hind_left_collision"]
        self.JOINT_POS_SENSOR = "joint_pos"
        self.IMU_GYRO_SENSOR = "imu_gyro"
        self.TORSO_QUAT_SENSOR = "torso_quat"
        self.JOINT_VEL_SENSOR = "joint_vel"
        self.ACTUATOR_FORCES_SENSOR = "actuator_forces"
        self.FOOT_CONTACTS_SENSOR = "foot_contacts"

consts = _PupperConstants()
print("Pupper 常數模組已定義。")


# --- Pupper 預設配置 ---
def get_default_config() -> config_dict.ConfigDict:
    config = config_dict.ConfigDict()
    config.sim_dt = 0.002; config.ctrl_dt = 0.02
    config.episode_length = 1000; config.action_repeat = 10
    config.action_scale = 0.3
    config.soft_joint_pos_limit_factor = 0.95
    reward_scales = {
        'tracking_lin_vel': 1.5, 'tracking_ang_vel': 0.8, 'lin_vel_z': -2.0,
        'ang_vel_xy': -0.05, 'orientation': -5.0, 'torques': -0.0002,
        'action_rate': -0.01,
    }
    config.rewards = config_dict.ConfigDict({'scales': config_dict.ConfigDict(reward_scales), 'tracking_sigma': 0.25})
    config.commands = config_dict.ConfigDict({'ranges': {'lin_vel_x': [-0.6, 1.5], 'lin_vel_y': [-0.8, 0.8], 'ang_vel_yaw': [-0.7, 0.7]}, 'zero_command_probability': 0.1})
    return config

print("預設配置函式 get_default_config() 已定義。")

Pupper 常數模組已定義。


In [85]:
#@title Pupper 預設配置

def get_default_config() -> config_dict.ConfigDict:
    """返回 Pupper 四足機器人環境的預設配置。"""
    config = config_dict.ConfigDict()
    config.sim_dt = 0.002; config.ctrl_dt = 0.02
    config.episode_length = 1000; config.action_repeat = 10
    config.Kp = 35.0; config.Kd = 0.5
    config.action_scale = 0.3; config.history_len = 3
    config.soft_joint_pos_limit_factor = 0.95
    config.noise_config = config_dict.ConfigDict({'level': 0.05, 'scales': {'joint_pos': 0.03, 'gyro': 0.2}})
    reward_scales = {
        'tracking_lin_vel': 1.5, 'tracking_ang_vel': 0.8, 'lin_vel_z': -2.0,
        'ang_vel_xy': -0.05, 'orientation': -5.0, 'torques': -0.0002,
        'action_rate': -0.01, 'energy': -0.001, 'dof_pos_limits': -1.0,
        'stand_still': -0.5, 'pose': 0.5, 'termination': -1.0,
        'foot_slip': -0.1, 'feet_air_time': 0.2, 'feet_clearance': -2.0, 'feet_height': 0.2,
    }
    config.rewards = config_dict.ConfigDict({'scales': config_dict.ConfigDict(reward_scales), 'tracking_sigma': 0.25, 'max_foot_height': 0.1})
    config.perturbation = config_dict.ConfigDict({'enable': True, 'velocity_kick': [0.0, 0.05], 'kick_durations': [0.05, 0.2], 'kick_wait_times': [1.0, 3.0]})
    config.commands = config_dict.ConfigDict({'ranges': {'lin_vel_x': [-0.6, 1.5], 'lin_vel_y': [-0.8, 0.8], 'ang_vel_yaw': [-0.7, 0.7]}, 'zero_command_probability': 0.1})
    return config

print("預設配置函式 get_default_config() 已定義。")

預設配置函式 get_default_config() 已定義。


In [86]:
#@title Pupper 基礎環境 (PupperEnv) - 最終架構

# 直接繼承自 Brax 的基礎 Env 類別
class PupperEnv(Env):
  """
  Pupper 模型的基礎環境，處理模型載入和感測器資料介面。
  """
  def __init__(self, xml_path: str, config: config_dict.ConfigDict, **kwargs):
    # --- 1. 模型載入流程 ---
    mj_model = mujoco.MjModel.from_xml_path(xml_path)
    # 處理 mesh 檔案路徑
    for i, path in enumerate(mj_model.mesh_fpath):
        mj_model.mesh_fpath[i] = os.path.join(consts.PUPPER_ROOT_PATH, path)
    sys = mjcf.load_model(mj_model)

    # ========================== 核心修正 ==========================
    # 在擁有 `sys` 物件後，立刻呼叫父類別 (brax.envs.base.Env) 的建構函式。
    # `backend='mjx'` 等參數會被包含在 **kwargs 中並被正確傳遞。
    super().__init__(sys=sys, **kwargs)
    # =============================================================

    # --- 2. 在 super().__init__ 後設定其他自訂屬性 ---
    self._mj_model = mj_model # 方便存取原始 mjModel
    self.model = self._mj_model # 為了與舊 API 相容

    # 儲存配置
    self._config = config
    self._config.merge(kwargs) # 允許 `get_environment` 的額外參數覆蓋預設配置
    
    # 從配置中設定步進參數
    self.dt = self._config.ctrl_dt
    self.n_substeps = self._config.action_repeat

    # --- 3. 建立感測器索引 ---
    self._sensor_indices = {
        'joint_pos': mjcf.get_sensor(mj_model, consts.JOINT_POS_SENSOR),
        'imu_accel': mjcf.get_sensor(mj_model, consts.IMU_ACCEL_SENSOR),
        'imu_gyro': mjcf.get_sensor(mj_model, consts.IMU_GYRO_SENSOR),
        'torso_quat': mjcf.get_sensor(mj_model, consts.TORSO_QUAT_SENSOR),
        'joint_vel': mjcf.get_sensor(mj_model, consts.JOINT_VEL_SENSOR),
        'actuator_forces': mjcf.get_sensor(mj_model, consts.ACTUATOR_FORCES_SENSOR),
        'foot_contacts': mjcf.get_sensor(mj_model, consts.FOOT_CONTACTS_SENSOR),
    }

  # 因為不再繼承自虛擬的 MjxEnv，我們需要自己實現 pipeline_step
  def pipeline_step(self, data: mjx.Data, ctrl: jax.Array) -> mjx.Data:
    def f(data, _):
        data = data.replace(ctrl=ctrl)
        # self.sys 是由 `super().__init__` 設定的 Brax System 物件
        return mjx.step(self.sys, data), None
    data, _ = jax.lax.scan(f, data, (), self.n_substeps)
    return data

  def _read_sensor(self, data: mjx.Data, name: str) -> jax.Array:
    idx = self._sensor_indices[name]
    return data.sensordata[idx.adr:idx.adr + idx.dim]

  # --- 感測器資料讀取輔助函式 ---
  def get_joint_pos(self, data: mjx.Data) -> jax.Array: return self._read_sensor(data, 'joint_pos')
  def get_joint_vel(self, data: mjx.Data) -> jax.Array: return self._read_sensor(data, 'joint_vel')
  def get_actuator_forces(self, data: mjx.Data) -> jax.Array: return self._read_sensor(data, 'actuator_forces')
  def get_foot_contacts(self, data: mjx.Data) -> jax.Array: return self._read_sensor(data, 'foot_contacts')
  def get_gyro(self, data: mjx.Data) -> jax.Array: return self._read_sensor(data, 'imu_gyro')
  
  def get_projected_gravity(self, data: mjx.Data) -> jax.Array:
    torso_quat = self._read_sensor(data, 'torso_quat')
    return math.rotate(jp.array([0, 0, -1]), torso_quat)

  @property
  def action_size(self) -> int:
    return self.sys.nu

print("PupperEnv 基礎類別已使用最終架構重新定義。")

PupperEnv 基礎類別已使用最終架構重新定義。


In [87]:
#@title Pupper 搖桿任務環境 (PupperJoystick) - 最終修正版

class PupperJoystick(PupperEnv):
  """
  Pupper 的具體任務環境：追蹤一個由搖桿生成的、隨時間變化的指令。
  """
  # ========================== 核心修正 ==========================
  # 簡化 __init__，僅 pop 本層需要的參數，其餘用 **kwargs 傳遞
  def __init__(self, **kwargs):
    config = kwargs.pop('config', get_default_config())
    xml_path = kwargs.pop('xml_path', consts.DEFAULT_XML.as_posix())
    
    # 將 xml_path, config 和剩餘的 kwargs 傳遞給父類別
    super().__init__(xml_path=xml_path, config=config, **kwargs)

    self._post_init()
  # =============================================================

  def _post_init(self):
    """在建構函式之後執行，用於初始化任務相關的變數。"""
    self._init_q = jp.array(self._mj_model.keyframe("home").qpos)
    self._default_pose = self._init_q[7:]
    
    jnt_range = self.mj_model.jnt_range.T
    self._lowers, self._uppers = jnt_range[:, 6:]
    self._soft_lowers = self._lowers * self._config.soft_joint_pos_limit_factor
    self._soft_uppers = self._uppers * self._config.soft_joint_pos_limit_factor

    self._torso_body_id = mujoco.mj_name2id(self._mj_model, mujoco.mjtObj.mjOBJ_BODY, consts.ROOT_BODY)
    self._floor_geom_id = mujoco.mj_name2id(self._mj_model, mujoco.mjtObj.mjOBJ_GEOM, "floor")
    self._feet_geom_id = np.array([mujoco.mj_name2id(self._mj_model, mujoco.mjtObj.mjOBJ_GEOM, n) for n in consts.FEET_GEOMS])
    
    self._cmd_ranges = self._config.commands.ranges
    self._zero_cmd_prob = self._config.commands.zero_command_probability

  def reset(self, rng: jax.Array) -> State:
    rng, key_vel, key_cmd = jax.random.split(rng, 3)
    qpos = self._init_q
    qvel = jax.random.uniform(key_vel, (self.mjx_model.nv,), minval=-0.1, maxval=0.1)
    
    data = mjx.make_data(self.mjx_model).replace(qpos=qpos, qvel=qvel)
    data = mjx.step(self.mjx_model, data)
    
    cmd = self.sample_command(key_cmd)
    info = {"rng": rng, "command": cmd, "last_act": jp.zeros(self.action_size), "feet_air_time": jp.zeros(4), "last_contact": jp.zeros(4, dtype=bool)}
    obs_dict = self._get_obs(data, info)
    reward, done = jp.zeros(2)
    metrics = {f"reward/{k}": jp.zeros(()) for k in self._config.rewards.scales.keys()}
    return State(pipeline_state=data, obs=obs_dict, reward=reward, done=done, metrics=metrics, info=info)

  def step(self, state: State, action: jax.Array) -> State:
    data = state.pipeline_state
    motor_targets = self._default_pose + action * self._config.action_scale
    data = self.pipeline_step(data, motor_targets)
    
    def _is_in_contact(geom_id, contacts):
        return jp.any(contacts.geom1 == geom_id)
    
    floor_contacts = data.contact[data.contact.geom2 == self._floor_geom_id]
    contact = jax.vmap(_is_in_contact, in_axes=(0, None))(self._feet_geom_id, floor_contacts)

    rng, key_cmd = jax.random.split(state.info['rng'])
    command = self.sample_command(key_cmd)
    info = state.info.copy()
    info.update(rng=rng, command=command, last_act=action,
                feet_air_time=(state.info["feet_air_time"] + self.dt) * (1 - contact),
                last_contact=contact)
                
    obs_dict = self._get_obs(data, info)
    rewards_dict = self._get_reward(data, action, info)
    reward = sum(v * self._config.rewards.scales.get(k, 0.0) for k, v in rewards_dict.items())
    done = self._get_termination(data)
    
    metrics = state.metrics.copy()
    metrics.update({f"reward/{k}": v for k, v in rewards_dict.items()})
    return State(pipeline_state=data, obs=obs_dict, reward=reward, done=done.astype(reward.dtype), metrics=metrics, info=info)

  def _get_obs(self, data: mjx.Data, info: dict) -> Dict[str, jax.Array]:
    joint_angles = self.get_joint_pos(data); joint_vel = self.get_joint_vel(data)
    standard_obs = jp.concatenate([
        self.get_gyro(data), self.get_projected_gravity(data),
        info['command'], joint_angles - self._default_pose, info['last_act']
    ])
    privileged_obs = jp.concatenate([
        standard_obs, joint_vel, self.get_actuator_forces(data), self.get_foot_contacts(data)
    ])
    return {'state': standard_obs, 'privileged_state': privileged_obs}

  def _get_reward(self, data: mjx.Data, action: jax.Array, info: dict) -> dict:
      rewards = {}
      torso_vel = data.cvel[self._torso_body_id]
      torso_quat = data.xquat[self._torso_body_id]
      local_vel = math.rotate(torso_vel[:3], math.quat_inv(torso_quat))
      lin_vel_error = jp.sum(jp.square(info['command'][:2] - local_vel[:2]))
      rewards['tracking_lin_vel'] = jp.exp(-lin_vel_error / self._config.rewards.tracking_sigma)
      ang_vel_error = jp.square(info['command'][2] - self.get_gyro(data)[2])
      rewards['tracking_ang_vel'] = jp.exp(-ang_vel_error / self._config.rewards.tracking_sigma)
      rewards['orientation'] = -jp.sum(jp.square(self.get_projected_gravity(data)[:2]))
      rewards['lin_vel_z'] = -jp.square(local_vel[2])
      rewards['ang_vel_xy'] = -jp.sum(jp.square(self.get_gyro(data)[:2]))
      rewards['torques'] = -jp.sum(jp.square(self.get_actuator_forces(data)))
      rewards['action_rate'] = -jp.sum(jp.square(action - info['last_act']))
      return rewards

  def _get_termination(self, data: mjx.Data) -> jax.Array:
    return data.xmat[self._torso_body_id][2, 2] < 0.3

  def sample_command(self, rng: jax.Array) -> jax.Array:
    rng, key_x, key_y, key_z, key_zero = jax.random.split(rng, 5)
    lin_vel_x = jax.random.uniform(key_x, minval=self._cmd_ranges.lin_vel_x[0], maxval=self._cmd_ranges.lin_vel_x[1])
    lin_vel_y = jax.random.uniform(key_y, minval=self._cmd_ranges.lin_vel_y[0], maxval=self._cmd_ranges.lin_vel_y[1])
    ang_vel_yaw = jax.random.uniform(key_z, minval=self._cmd_ranges.ang_vel_yaw[0], maxval=self._cmd_ranges.ang_vel_yaw[1])
    command = jp.array([lin_vel_x, lin_vel_y, ang_vel_yaw])
    is_zero = jax.random.bernoulli(key_zero, self._zero_cmd_prob)
    return jp.where(is_zero, jp.zeros(3), command)

  @property
  def observation_size(self) -> Dict[str, Tuple[int, ...]]:
    return {"state": (33,), "privileged_state": (61,)}

# 註冊環境
if 'pupper_joystick' in envs._envs: del envs._envs['pupper_joystick']
envs.register_environment('pupper_joystick', PupperJoystick)
print("PupperJoystick 任務環境已修正並重新註冊。")

PupperJoystick 任務環境已修正並重新註冊。


In [88]:
#@title 訓練主程式

# --- 1. 初始化環境 ---
# 直接使用我們註冊的、返回字典觀測的原始環境
env = envs.get_environment('pupper_joystick')
eval_env = envs.get_environment('pupper_joystick')
print(f"環境 '{type(env).__name__}' 已成功初始化。")


# --- 2. 定義網路工廠 (Network Factory) ---
# 這個工廠函式告訴 PPO 如何創建神經網路。
# 關鍵在於指定 policy_observation_key 和 value_observation_key，
# 這啟用了非對稱 Actor-Critic 模式。
make_networks_factory = functools.partial(
    ppo_networks.make_ppo_networks,
    policy_hidden_layer_sizes=(512, 256, 128),
    value_hidden_layer_sizes=(512, 256, 128),
    policy_observation_key='state',
    value_observation_key='privileged_state'
)
print("非對稱 Actor-Critic 網路工廠已定義。")


# --- 3. 定義 PPO 訓練函式 ---
# 我們在這裡集中設定所有 PPO 的超參數。
train_fn = functools.partial(
    ppo.train,
    # --- 核心訓練參數 ---
    num_timesteps=100_000_000,   # 總訓練步數
    num_envs=2048,              # 並行環境的數量 (可根據 GPU VRAM 調整)
    episode_length=1000,        # 每個回合的最大長度
    
    # --- PPO 演算法參數 ---
    normalize_observations=True,# 對觀測進行正規化，非常重要！
    unroll_length=20,           # 每次採樣的序列長度
    num_minibatches=32,         # 將一個批次的數據分成多少個小批次
    num_updates_per_batch=8,    # 每次採樣後，用這些數據更新網路的次數
    batch_size=1024,            # 每個小批次的大小
    
    # --- 獎勵與學習率 ---
    discounting=0.99,           # 折扣因子 (gamma)，更看重長期回報
    learning_rate=3e-4,         # 學習率
    entropy_cost=0.01,          # 熵成本，鼓勵探索
    
    # --- 其他 ---
    network_factory=make_networks_factory,
    seed=0,
    num_evals=20, # 訓練過程中進行多少次評估
)
print("PPO 訓練函式已配置完成。")


# --- 4. 啟動訓練並視覺化進度 ---
x_data, y_data = [], []
times = [datetime.now()]

def progress(num_steps, metrics):
    """一個在訓練過程中被呼叫的回呼函式，用於繪製獎勵曲線。"""
    times.append(datetime.now())
    x_data.append(num_steps)
    y_data.append(metrics['eval/episode_reward'])
    
    # 清除舊的輸出並繪製新圖
    clear_output(wait=True)
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.figure(figsize=(10, 6))
    plt.plot(x_data, y_data)
    plt.xlabel('Environment Steps')
    plt.ylabel('Average Episode Reward')
    plt.title(f"Training Progress - Current Reward: {y_data[-1]:.2f}")
    plt.show()

print("\n--- 即將開始訓練 ---")

make_inference_fn, params, _ = train_fn(
    environment=env,
    eval_env=eval_env,
    progress_fn=progress
)

print(f"\n--- 訓練完成！---")
print(f"總耗時: {times[-1] - times[0]}")

TypeError: Can't instantiate abstract class PupperJoystick without an implementation for abstract method 'backend'